# UrbanEats Delivery Operations — Notebook 2
## Part C1: Cancellation Risk Classifier | Part C2: Ops Alerts with LangChain + Groq

**Assignment 2 · FDE Masterclass**  
Domain: Food Delivery & Operations Analytics

---
## Part C1 — Cancellation Risk Classifier

In [1]:
# Step 10: Load data and prepare features
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('/content/urbaneats_delivery_orders.csv')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} columns')

# ── Preprocessing: fill missing delivery_time_mins with ZONE MEDIAN (before split)
# This is done on the full 150-row dataset first, then we carve out the model subset
zone_medians = df.groupby('delivery_zone')['delivery_time_mins'].median()
print('\nZone medians for delivery_time_mins imputation:')
print(zone_medians.to_dict())

df['delivery_time_mins'] = df.apply(
    lambda r: zone_medians[r['delivery_zone']] if pd.isna(r['delivery_time_mins']) else r['delivery_time_mins'],
    axis=1
)
print(f'\nMissing delivery_time_mins after imputation: {df["delivery_time_mins"].isnull().sum()}')

# Label-encode categoricals on FULL dataset (ensures consistent encoding for scoring all 150)
from sklearn.preprocessing import LabelEncoder
le_zone = LabelEncoder()
le_rest = LabelEncoder()
df['zone_enc'] = le_zone.fit_transform(df['delivery_zone'])
df['rest_enc'] = le_rest.fit_transform(df['restaurant_name'])

print('\nZone encoding map:', dict(zip(le_zone.classes_, le_zone.transform(le_zone.classes_))))
print('Restaurant encoding map:', dict(zip(le_rest.classes_, le_rest.transform(le_rest.classes_))))

Loaded: 150 rows × 11 columns

Zone medians for delivery_time_mins imputation:
{'Central': 60.0, 'East': 53.5, 'North': 54.5, 'South': 46.0, 'West': 57.0}

Missing delivery_time_mins after imputation: 0

Zone encoding map: {'Central': np.int64(0), 'East': np.int64(1), 'North': np.int64(2), 'South': np.int64(3), 'West': np.int64(4)}
Restaurant encoding map: {'Burger Hub': np.int64(0), 'Pizza Palace': np.int64(1), 'Spice Garden': np.int64(2), 'Sushi Bay': np.int64(3), 'Wrap & Roll': np.int64(4)}


In [2]:
# Step 10 (cont.): Filter to Delivered + Cancelled; build target variable
# Per assignment: use ONLY Delivered/Cancelled rows for training the classifier
df_model = df[df['order_status'].isin(['Delivered', 'Cancelled'])].copy()
df_model['target'] = (df_model['order_status'] == 'Cancelled').astype(int)

print(f'Model training subset: {len(df_model)} rows (Delivered={len(df_model[df_model["target"]==0])}, Cancelled={len(df_model[df_model["target"]==1])})')
print(f'Full dataset retained for scoring: {len(df)} rows')

# Summary statistics for all 4 statuses (included in summary, excluded from modelling)
print('\nFull dataset order_status distribution:')
print(df['order_status'].value_counts())

Model training subset: 80 rows (Delivered=44, Cancelled=36)
Full dataset retained for scoring: 150 rows

Full dataset order_status distribution:
order_status
Delivered    44
Refunded     37
Cancelled    36
Delayed      33
Name: count, dtype: int64


In [3]:
# Step 11: Train Random Forest (n_estimators=100, random_state=42) on 80/20 split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report, confusion_matrix

FEATURES = ['zone_enc', 'rest_enc', 'order_value', 'discount_applied', 'delivery_time_mins']

X = df_model[FEATURES]
y = df_model['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Train size: {len(X_train)} | Test size: {len(X_test)}')

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print('\n✅ Random Forest trained successfully')

Train size: 64 | Test size: 16

✅ Random Forest trained successfully


In [4]:
# Step 12: Print precision, recall, F1 — and feature importance analysis
precision = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
f1        = f1_score(y_test, y_pred, zero_division=0)

print('=== Model Evaluation (Test Set) ===')
print(f'  Precision : {precision:.3f}')
print(f'  Recall    : {recall:.3f}')
print(f'  F1-Score  : {f1:.3f}')
print()
print(classification_report(y_test, y_pred, target_names=['Delivered', 'Cancelled']))

print('\n=== Feature Importances ===')
importance_df = pd.DataFrame({
    'feature': FEATURES,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
print(importance_df.to_string(index=False))

=== Model Evaluation (Test Set) ===
  Precision : 0.600
  Recall    : 0.375
  F1-Score  : 0.462

              precision    recall  f1-score   support

   Delivered       0.55      0.75      0.63         8
   Cancelled       0.60      0.38      0.46         8

    accuracy                           0.56        16
   macro avg       0.57      0.56      0.55        16
weighted avg       0.57      0.56      0.55        16


=== Feature Importances ===
           feature  importance
delivery_time_mins    0.334411
  discount_applied    0.209335
       order_value    0.195033
          zone_enc    0.153572
          rest_enc    0.107649


## C1 — Classifier Interpretation

**Highest-risk restaurant-zone combination based on model outputs:**

The top predictors (in descending order of contribution) are:
1. `delivery_time_mins` — 33.4% of predictive weight
2. `discount_applied` — 20.9%
3. `order_value` — 19.5%
4. `zone_enc` (delivery zone) — 15.4%
5. `rest_enc` (restaurant) — 10.8%

After scoring all 150 orders and grouping by restaurant-zone pairs, the highest cancellation risk combination is **Wrap & Roll in the Central zone** (75% of its orders are high-risk), followed closely by **Burger Hub in the Central zone** (70% high-risk rate).

**Business implication:**  
> The Central delivery zone, particularly for Wrap & Roll and Burger Hub, is a critical intervention point — long delivery times (averaging 53–67 minutes) coupled with high discounts are triggering a self-reinforcing cancellation cycle that, if unaddressed, will continue to erode revenue and inflate the refund liability.

In [5]:
# Step 13: Score ALL 150 orders — add cancel_probability and cancel_risk columns
X_all = df[FEATURES]

df['cancel_probability'] = rf.predict_proba(X_all)[:, 1].round(4)
df['cancel_risk'] = df['cancel_probability'].apply(lambda x: 'high' if x >= 0.55 else 'low')

print(f'Scored all {len(df)} orders')
print(f'High cancel_risk (>= 0.55): {(df["cancel_risk"] == "high").sum()} orders')
print(f'Low cancel_risk:             {(df["cancel_risk"] == "low").sum()} orders')
print()
print('Cancel risk distribution by order_status:')
print(df.groupby(['order_status', 'cancel_risk']).size().unstack(fill_value=0))

Scored all 150 orders
High cancel_risk (>= 0.55): 49 orders
Low cancel_risk:             101 orders

Cancel risk distribution by order_status:
cancel_risk   high  low
order_status           
Cancelled       30    6
Delayed          9   24
Delivered        2   42
Refunded         8   29


In [6]:
# Step 14–15: Group analysis — identify hotspots where high_risk_rate > 30%
group_df = df.groupby(['restaurant_name', 'delivery_zone']).agg(
    total_orders      = ('order_id', 'count'),
    high_risk_count   = ('cancel_risk', lambda x: (x == 'high').sum()),
    avg_order_value   = ('order_value', 'mean'),
    avg_delivery_time = ('delivery_time_mins', 'mean')
).reset_index()

group_df['high_risk_rate'] = (group_df['high_risk_count'] / group_df['total_orders']).round(3)
group_df['avg_order_value'] = group_df['avg_order_value'].round(1)
group_df['avg_delivery_time'] = group_df['avg_delivery_time'].round(1)

hotspots = group_df[group_df['high_risk_rate'] > 0.30].sort_values('high_risk_rate', ascending=False)

print(f'Restaurant-zone groups with high_risk_rate > 30%: {len(hotspots)}')
print()
print(hotspots[['restaurant_name', 'delivery_zone', 'high_risk_rate',
                'high_risk_count', 'total_orders', 'avg_order_value', 'avg_delivery_time']].to_string(index=False))

Restaurant-zone groups with high_risk_rate > 30%: 13

restaurant_name delivery_zone  high_risk_rate  high_risk_count  total_orders  avg_order_value  avg_delivery_time
    Wrap & Roll       Central           0.750                3             4            818.2               53.5
     Burger Hub       Central           0.700                7            10           1196.3               67.3
    Wrap & Roll          West           0.600                3             5            595.4               63.6
   Spice Garden          West           0.571                4             7           1093.7               58.0
   Pizza Palace       Central           0.500                1             2           1258.0               60.0
   Spice Garden       Central           0.500                2             4            861.0               41.2
    Wrap & Roll          East           0.500                2             4           1063.5               64.6
      Sushi Bay       Central           0.

---
## Part C2 — Ops Alert Messages with LangChain + Groq

In [7]:
# Step 16–17: LangChain + Groq — generate ops manager alerts

# Ensure latest versions are installed
!pip install -U langchain-groq langchain-core langchain

import os
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from google.colab import userdata

try:
    # Retrieve Groq API key from Colab Secrets
    # Note: Ensure you have added 'GROQ_API_KEY' to the Secrets tab (key icon) on the left
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

    # Initialise Groq (llama-3.1-8b-instant)
    llm = ChatGroq(
        model='llama-3.1-8b-instant',
        max_tokens=512,
        temperature=0.3
    )

    # PromptTemplate with required variables
    alert_template = PromptTemplate(
        input_variables=['restaurant_name', 'delivery_zone', 'high_risk_rate',
                         'avg_order_value', 'avg_delivery_time'],
        template="""You are an operations intelligence system for UrbanEats, a food delivery platform.
Write a 2-sentence ops alert for a regional manager.

Hotspot data:
- Restaurant: {restaurant_name}
- Delivery Zone: {delivery_zone}
- High Cancellation Risk Rate: {high_risk_rate}%
- Average Order Value: ₹{avg_order_value}
- Average Delivery Time: {avg_delivery_time} minutes

Requirements:
1. Sentence 1: Name the exact restaurant-zone combination and state the cancellation risk rate.
2. Sentence 2: Recommend ONE specific, concrete operational action.
3. No hedging language.
4. Be direct, specific, and action-oriented.

Alert:"""
    )

    # Use LCEL (LangChain Expression Language) with the correct parser
    chain = alert_template | llm | StrOutputParser()

    print('✅ LangChain LCEL Chain initialised successfully')
    print('Model: llama-3.1-8b-instant (Groq)')

except userdata.SecretNotFoundError:
    print('❌ ERROR: GROQ_API_KEY not found in Colab Secrets.')
    print('Please click the key icon on the left sidebar, add "GROQ_API_KEY", and enable notebook access.')
except Exception as e:
    print(f'❌ ERROR: {e}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.3 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.3
    Uninstalling langchain-core-1.4.3:
      Successfully uninstalled langchain-core-1.4.3
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.6
    Uninstalling langchain-1.3.6:
      Successfully uninstalled langchain-1.3.6
✅ LangChain LCEL Chain initialised successfully
Model: llama-3.1-8b-instant (Groq)


In [8]:
# Generate alerts for all hotspots using the LCEL chain
alerts = []

for _, row in hotspots.iterrows():
    # chain.invoke() takes a dict matching the PromptTemplate input_variables
    alert_text = chain.invoke({
        "restaurant_name":   row["restaurant_name"],
        "delivery_zone":     row["delivery_zone"],
        "high_risk_rate":    round(row["high_risk_rate"] * 100, 1),
        "avg_order_value":   row["avg_order_value"],
        "avg_delivery_time": row["avg_delivery_time"]
    })
    alert_text = alert_text.strip()

    alerts.append({
        "restaurant_name":   row["restaurant_name"],
        "delivery_zone":     row["delivery_zone"],
        "high_risk_rate":    row["high_risk_rate"],
        "avg_order_value":   row["avg_order_value"],
        "avg_delivery_time": row["avg_delivery_time"],
        "total_orders":      row["total_orders"],
        "alert_text":        alert_text
    })

    rname = row["restaurant_name"]
    rzone = row["delivery_zone"]
    rrate = row["high_risk_rate"] * 100
    print(f"🔴 {rname} — {rzone} (risk: {rrate:.1f}%)")
    print(f"   {alert_text}")
    print()


🔴 Wrap & Roll — Central (risk: 75.0%)
   "Wrap & Roll in Central has a high cancellation risk rate of 75.0%. We recommend that you implement a proactive communication strategy with the restaurant to address the root cause of cancellations and provide incentives to reduce last-minute order cancellations."

🔴 Burger Hub — Central (risk: 70.0%)
   Alert: Burger Hub in Central Delivery Zone has a high cancellation risk rate of 70.0%. Implement a 10-minute call-back policy for all orders from Burger Hub in Central to proactively address customer concerns and reduce cancellations.

🔴 Wrap & Roll — West (risk: 60.0%)
   Alert: Wrap & Roll in the West delivery zone is experiencing a high cancellation risk rate of 60.0%. 

Recommendation: Immediately conduct a site visit to Wrap & Roll in the West to assess and address any operational issues contributing to the high cancellation rate, and provide coaching to the restaurant staff on order management and communication protocols.

🔴 Spice Garden —

In [9]:
# Step 17: Evaluate 3 alerts — specificity, actionability, no hedging
def evaluate_alert(alert):
    text = alert['alert_text'].lower()
    restaurant = alert['restaurant_name'].lower()
    zone = alert['delivery_zone'].lower()

    # 1. Specificity: names the exact hotspot
    specificity = (restaurant in text) and (zone in text)

    # 2. Actionability: contains concrete action words
    action_words = ['increase', 'reduce', 'assign', 'audit', 'review', 'deploy',
                    'add', 'dispatch', 'allocate', 'prioritise', 'prioritize',
                    'escalate', 'investigate', 'schedule', 'limit', 'cap']
    actionability = any(w in text for w in action_words)

    # 3. No hedging language
    hedge_words = ['it seems', 'perhaps', 'might', 'could possibly', 'may want to',
                   'probably', 'appears to', 'seemingly']
    no_hedging = not any(h in text for h in hedge_words)

    return {'specificity': specificity, 'actionability': actionability, 'no_hedging': no_hedging}

print('=== Alert Quality Evaluation (3 criteria) ===')
print(f'{"Restaurant-Zone":35s} | {"Specificity":12s} | {"Actionability":14s} | {"No Hedging":10s}')
print('-' * 80)

for i, alert in enumerate(alerts[:3]):  # Evaluate first 3
    scores = evaluate_alert(alert)
    combo = f'{alert["restaurant_name"]} — {alert["delivery_zone"]}'
    print(f'{combo:35s} | {"✅" if scores["specificity"] else "❌":12s} | {"✅" if scores["actionability"] else "❌":14s} | {"✅" if scores["no_hedging"] else "❌":10s}')

print('\n--- Full Alert Texts (first 3) ---')
for alert in alerts[:3]:
    print(f'\n🔴 {alert["restaurant_name"]} | {alert["delivery_zone"]} | Risk: {alert["high_risk_rate"]*100:.0f}%')
    print(f'   {alert["alert_text"]}')

=== Alert Quality Evaluation (3 criteria) ===
Restaurant-Zone                     | Specificity  | Actionability  | No Hedging
--------------------------------------------------------------------------------
Wrap & Roll — Central               | ✅            | ✅              | ✅         
Burger Hub — Central                | ✅            | ✅              | ✅         
Wrap & Roll — West                  | ✅            | ✅              | ✅         

--- Full Alert Texts (first 3) ---

🔴 Wrap & Roll | Central | Risk: 75%
   "Wrap & Roll in Central has a high cancellation risk rate of 75.0%. We recommend that you implement a proactive communication strategy with the restaurant to address the root cause of cancellations and provide incentives to reduce last-minute order cancellations."

🔴 Burger Hub | Central | Risk: 70%
   Alert: Burger Hub in Central Delivery Zone has a high cancellation risk rate of 70.0%. Implement a 10-minute call-back policy for all orders from Burger Hub in Central t

In [10]:
# Save the enriched CSV with cancel_probability and cancel_risk columns
output_cols = [
    'order_id', 'order_date', 'restaurant_name', 'delivery_zone',
    'order_value', 'delivery_time_mins', 'rider_rating', 'order_status',
    'payment_method', 'discount_applied', 'customer_complaints',
    'cancel_probability', 'cancel_risk'
]
df_output = df[output_cols].copy()
df_output.to_csv('urbaneats_delivery_orders_scored.csv', index=False)
print(f'✅ Scored CSV saved → urbaneats_delivery_orders_scored.csv')
print(f'   Rows: {len(df_output)} | New columns: cancel_probability, cancel_risk')
print()
print('Sample (first 5 rows):')
df_output[['order_id', 'restaurant_name', 'delivery_zone', 'order_status',
           'cancel_probability', 'cancel_risk']].head(10)

✅ Scored CSV saved → urbaneats_delivery_orders_scored.csv
   Rows: 150 | New columns: cancel_probability, cancel_risk

Sample (first 5 rows):


,order_id,restaurant_name,delivery_zone,order_status,cancel_probability,cancel_risk
0,ORD00001,Pizza Palace,North,Delayed,0.32,low
1,ORD00002,Pizza Palace,East,Cancelled,0.24,low
2,ORD00003,Spice Garden,South,Delayed,0.53,low
3,ORD00004,Wrap & Roll,East,Delivered,0.31,low
4,ORD00005,Wrap & Roll,North,Refunded,0.33,low
5,ORD00006,Pizza Palace,West,Delivered,0.09,low
6,ORD00007,Pizza Palace,East,Cancelled,0.86,high
7,ORD00008,Burger Hub,North,Cancelled,0.54,low
8,ORD00009,Sushi Bay,East,Delivered,0.16,low
9,ORD00010,Sushi Bay,West,Delivered,0.07,low


In [11]:
# Save alerts summary for n8n workflow consumption
import json

alerts_summary = {
    'top_hotspot': {
        'restaurant_name':   alerts[0]['restaurant_name'],
        'delivery_zone':     alerts[0]['delivery_zone'],
        'high_risk_rate':    alerts[0]['high_risk_rate'],
        'avg_delivery_time': alerts[0]['avg_delivery_time'],
        'alert_text':        alerts[0]['alert_text']
    },
    'all_hotspots': alerts
}

with open('urbaneats_alerts.json', 'w') as f:
    json.dump(alerts_summary, f, indent=2)

print('✅ Alerts JSON saved → urbaneats_alerts.json')
print(f'   Total hotspots: {len(alerts)}')
print(f'   Worst hotspot:  {alerts[0]["restaurant_name"]} — {alerts[0]["delivery_zone"]} ({alerts[0]["high_risk_rate"]*100:.0f}% risk)')

✅ Alerts JSON saved → urbaneats_alerts.json
   Total hotspots: 13
   Worst hotspot:  Wrap & Roll — Central (75% risk)
